In [7]:
from utils.spark_session import createSpark
from pyspark.sql import functions as F

spark = createSpark()

TOPIC = "crypto.tickers"
CHECKPOINT_BRONZE_PATH = f"s3a://spark-checkpoints/bronze/{TOPIC}"
CHECKPOINT_SILVER_PATH = f"s3a://spark-checkpoints/silver/{TOPIC}"
BRONZE_PATH = f"s3a://crypto-lake/bronze/{TOPIC}"
SILVER_PATH = f"s3a://crypto-lake/silver/{TOPIC}"

df = spark.read.parquet(SILVER_PATH)
silver_schema = df.schema



In [8]:
df.printSchema()

root
 |-- ingestion_ts: timestamp (nullable = true)
 |-- kafka_ts: timestamp (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- type: string (nullable = true)
 |-- cs: long (nullable = true)
 |-- symbol: string (nullable = true)
 |-- last_price: decimal(18,8) (nullable = true)
 |-- high_price_24h: decimal(18,8) (nullable = true)
 |-- low_price_24h: decimal(18,8) (nullable = true)
 |-- prev_price_24h: decimal(18,8) (nullable = true)
 |-- volume_24h: decimal(22,4) (nullable = true)
 |-- turnover_24h: decimal(22,4) (nullable = true)
 |-- price_24h_pct: decimal(10,6) (nullable = true)
 |-- usd_index_price: decimal(18,8) (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)



In [18]:

df.groupBy(["year", "month", "day"])\
.agg(F.first("last_price").alias("open"), F.max("last_price").alias("high"),\
     F.min("last_price").alias("low"), F.last("last_price").alias("close"),\
     F.count("*").alias("trades_count"), F.last("volume_24h").alias("volume"),\
     F.sum("usd_index_price").alias("usd_index_price")).show(10)


[Stage 11:===================================================> (384 + 12) / 397]

+----+-----+---+--------------+--------------+--------------+--------------+------------+---------+-------------------+
|year|month|day|          open|          high|           low|         close|trades_count|   volume|    usd_index_price|
+----+-----+---+--------------+--------------+--------------+--------------+------------+---------+-------------------+
|2026|    4| 30|76141.40000000|76655.00000000|76011.00000000|76469.00000000|       41908|6855.3402|3195451156.35978900|
|2026|    4| 29|76230.60000000|76651.30000000|75988.40000000|76082.00000000|        6231|8954.9483| 475406455.81636400|
|2026|    5|  2|78479.10000000|78598.40000000|78166.50000000|78423.10000000|       29564|4034.7669|2316189674.78280100|
|2026|    5|  1|77061.30000000|77507.80000000|76884.90000000|77147.80000000|       27464|6139.7424|2119840159.30404800|
+----+-----+---+--------------+--------------+--------------+--------------+------------+---------+-------------------+



In [23]:
candles_1m = (df
    .withColumn("bucket_start", F.date_trunc("minute", F.col("event_time")))
    .groupBy("symbol", "bucket_start")
    .agg(
        F.min("event_time").alias("first_ts"),
        F.max("event_time").alias("last_ts"),
        F.max("last_price").alias("high"),
        F.min("last_price").alias("low"),
        F.count("*").alias("trades_count"),
        F.max("volume_24h").alias("volume_24h_snapshot"),
    )
)
candles_1m.orderBy("bucket_start").show()

[Stage 17:=====================================================>(395 + 2) / 397]

+-------+-------------------+--------------------+--------------------+--------------+--------------+------------+-------------------+
| symbol|       bucket_start|            first_ts|             last_ts|          high|           low|trades_count|volume_24h_snapshot|
+-------+-------------------+--------------------+--------------------+--------------+--------------+------------+-------------------+
|BTCUSDT|2026-04-29 14:54:00|2026-04-29 14:54:...|2026-04-29 14:54:...|76651.30000000|76622.40000000|          97|          8950.8301|
|BTCUSDT|2026-04-29 14:55:00|2026-04-29 14:55:...|2026-04-29 14:55:...|76624.10000000|76609.10000000|          99|          8946.6066|
|BTCUSDT|2026-04-29 14:56:00|2026-04-29 14:56:...|2026-04-29 14:56:...|76639.00000000|76604.80000000|          92|          8927.9580|
|BTCUSDT|2026-04-29 14:57:00|2026-04-29 14:57:...|2026-04-29 14:57:...|76604.80000000|76587.10000000|         116|          8927.9580|
|BTCUSDT|2026-04-29 14:58:00|2026-04-29 14:58:...|2026-

In [27]:
def make_candle(df, timedelta: str):
    return df\
    .withColumn("bucket_start", F.date_trunc(timedelta, F.col("event_time")))\
    .groupBy("symbol", "bucket_start")\
    .agg(
        F.min("event_time").alias("first_ts"),
        F.max("event_time").alias("last_ts"),
        F.max("last_price").alias("high"),
        F.min("last_price").alias("low"),
        F.count("*").alias("trades_count"),
        F.max("volume_24h").alias("volume_24h_snapshot"),
    )



[Stage 20:===================================================> (384 + 12) / 397]

+-------+-------------------+--------------------+--------------------+--------------+--------------+------------+-------------------+
| symbol|       bucket_start|            first_ts|             last_ts|          high|           low|trades_count|volume_24h_snapshot|
+-------+-------------------+--------------------+--------------------+--------------+--------------+------------+-------------------+
|BTCUSDT|2026-04-29 14:00:00|2026-04-29 14:54:...|2026-04-29 14:59:...|76651.30000000|76548.20000000|         606|          8950.8301|
|BTCUSDT|2026-04-29 15:00:00|2026-04-29 15:00:...|2026-04-29 15:39:...|76586.70000000|75988.40000000|        5625|          8974.5160|
|BTCUSDT|2026-04-30 08:00:00|2026-04-30 08:00:...|2026-04-30 08:59:...|76217.00000000|76026.80000000|        5963|          9777.6680|
|BTCUSDT|2026-04-30 07:00:00|2026-04-30 07:48:...|2026-04-30 07:59:...|76186.30000000|76072.00000000|        1767|          9754.5604|
|BTCUSDT|2026-05-02 16:00:00|2026-05-02 16:00:...|2026-